# Self-Improving AI Assistant Workshop

This notebook mirrors the working demo script in this repo. It extracts facts, stores them in Agent Memory Server, retrieves relevant memories, and uses them to generate a real assistant response.

## 1) Setup

Make sure Redis and AMS are running first, then execute the cells in order.

In [ ]:
import json
import os
import re
from uuid import uuid4

import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('.env')

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
AMS_URL = os.environ.get('AMS_URL', 'http://localhost:8000')
AMS_V1_URL = f"{AMS_URL.rstrip('/')}/v1"
USER_ID = os.environ.get('USER_ID', 'test_user')

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT = """You are a helpful assistant with access to a user's remembered preferences.
Use remembered facts when they are relevant, but do not mention internal tools or memory systems.
If the user is sharing preferences or personal facts, acknowledge them briefly and respond naturally.
If the user asks for scheduling help and you know a meeting-time preference, use it in your answer.
Do not invent preferences that are not present in the provided memories.
"""

print('AMS_URL =', AMS_URL)
print('USER_ID =', USER_ID)
print('OPENAI_API_KEY set =', bool(OPENAI_API_KEY))


## 2) Fact extraction

This uses JSON-mode extraction first, then falls back to a rule-based parser.

In [ ]:
EXTRACTION_PROMPT = """
Extract short JSON facts from the user message. Only output JSON. Keys: prefers_meetings_at, likes
User message: {message}
"""

def normalize_facts(facts: dict) -> dict:
    normalized = {}

    prefers = facts.get('prefers_meetings_at')
    if isinstance(prefers, str):
        prefers = prefers.strip().lower()
        if prefers in {'morning', 'afternoon', 'evening'}:
            normalized['prefers_meetings_at'] = prefers

    likes = facts.get('likes')
    if isinstance(likes, str):
        likes = [likes]
    if likes:
        normalized['likes'] = likes

    return normalized

def rule_based_extract(message: str) -> dict:
    text = message.lower()
    facts = {}

    if 'meeting' in text:
        for slot in ('morning', 'afternoon', 'evening'):
            if slot in text:
                facts['prefers_meetings_at'] = slot
                break

    likes = []
    for pattern in (
        r"\bi love ([a-zA-Z0-9 ,'-]+)",
        r"\bi like ([a-zA-Z0-9 ,'-]+)",
    ):
        match = re.search(pattern, message, re.IGNORECASE)
        if not match:
            continue
        for item in re.split(r',| and ', match.group(1)):
            cleaned = item.strip(' .')
            if cleaned:
                likes.append(cleaned.lower())

    if likes:
        facts['likes'] = likes

    return normalize_facts(facts)

def validate_against_message(message: str, facts: dict) -> dict:
    validated = dict(facts)
    if not re.search(r"\bi (love|like)\b", message, re.IGNORECASE):
        validated.pop('likes', None)
    return validated

async def extract_facts(message: str) -> dict:
    prompt = EXTRACTION_PROMPT.format(message=message)
    try:
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object'},
            max_tokens=200,
        )
        text = resp.choices[0].message.content or '{}'
        return validate_against_message(message, normalize_facts(json.loads(text)))
    except Exception:
        return rule_based_extract(message)


## 3) AMS helpers

These match the current AMS `/v1` API.

In [ ]:
def create_long_term_memories(user_id: str, memories: list):
    payload = {
        'memories': [
            {
                'id': memory.get('id', f'mem-{uuid4()}'),
                'user_id': user_id,
                **memory,
            }
            for memory in memories
        ]
    }
    r = requests.post(f"{AMS_V1_URL}/long-term-memory/", json=payload)
    r.raise_for_status()
    return r.json()

def search_long_term_memory(user_id: str, query: str):
    payload = {
        'text': query,
        'user_id': {'eq': user_id},
        'limit': 10,
    }
    r = requests.post(f"{AMS_V1_URL}/long-term-memory/search", json=payload)
    r.raise_for_status()
    return r.json()

def list_memories(user_id: str):
    payload = {
        'text': '',
        'user_id': {'eq': user_id},
        'limit': 100,
    }
    r = requests.post(f"{AMS_V1_URL}/long-term-memory/search", json=payload)
    r.raise_for_status()
    return r.json()

def delete_memory(memory_id: str):
    r = requests.delete(
        f"{AMS_V1_URL}/long-term-memory",
        params={'memory_ids': memory_id},
    )
    r.raise_for_status()
    return r.json()


## 4) End-to-end assistant

In [ ]:
def format_memory_value(value):
    if isinstance(value, list):
        return ', '.join(str(item) for item in value)
    return str(value)

async def handle_user_message(message: str) -> str:
    facts = await extract_facts(message)
    memories = []
    for key, value in facts.items():
        memories.append({
            'text': f"{key}: {format_memory_value(value)}",
            'memory_type': 'semantic',
            'metadata': {'source': 'conversation', 'fact_key': key},
        })

    if memories:
        create_long_term_memories(USER_ID, memories)

    retrieved = search_long_term_memory(USER_ID, message)
    memory_texts = [memory['text'] for memory in retrieved.get('memories', [])]
    memory_texts.extend(memory['text'] for memory in memories)
    mem_text = '\n'.join(dict.fromkeys(memory_texts))

    user_prompt = (
        'Relevant memories:\n'
        f"{mem_text or 'None'}\n\n"
        f"User message: {message}"
    )

    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': user_prompt},
            ],
            max_tokens=200,
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()
    except Exception:
        if 'prefers_meetings_at: morning' in mem_text and 'schedule' in message.lower():
            return 'You prefer morning meetings, so I suggest a 30-minute morning slot next week.'
        if mem_text:
            return f"I've noted these preferences for future responses: {mem_text}"
        return 'Thanks, I understand. What would you like to do next?'


## 5) Try it out

In [ ]:
sample_messages = [
    'I prefer morning meetings and I love hiking.',
    'Schedule a 30-minute sync next week.',
]

sample_messages


In [ ]:
import asyncio

async def run_demo():
    for message in sample_messages:
        print('USER:', message)
        response = await handle_user_message(message)
        print('ASSISTANT:', response)
        print('-' * 80)

# Uncomment to run interactively in Jupyter.
# await run_demo()


## 6) Inspect stored memories

In [ ]:
# Uncomment after running the demo.
# list_memories(USER_ID)
